# EDSS 취업 코호트 균형 패널 민감도 분석

## tl;dr

학교 구성을 기준일 구간 안에서 고정해도 결론은 유지된다. 2016→2020 설명적 취업자 비율 변화는 전체 표본 -3.5848%p, 균형 패널 -3.5646%p이며 차이는 0.0202%p다. 비교 가능한 9개 전환의 증감 방향은 모두 일치한다.

## Context & Methods

### Key Assumptions

- 2010–2013년 6월 1일 구간과 2014–2020년 12월 31일 구간을 따로 균형화한다.
- 학교 OpenID가 해당 구간의 모든 코호트에 있을 때만 균형 패널에 포함한다.
- 비율은 학교별 비율의 평균이 아니라 취업자 수 합계 ÷ 졸업학생 수 합계다.
- 파생 비율은 공식 취업률이 아니며 인과효과를 뜻하지 않는다.

In [1]:
import csv
import hashlib
import importlib.util
import json
from pathlib import Path

import duckdb
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SCRIPT = REPO_ROOT / 'scripts' / 'analyze_edss_employment_balanced_panel.py'
SPEC = importlib.util.spec_from_file_location('balanced_panel', SCRIPT)
balanced_panel = importlib.util.module_from_spec(SPEC)
SPEC.loader.exec_module(balanced_panel)
DB = REPO_ROOT / 'data/processed/edss/restricted/edss_all.duckdb'
CSV_PATH = REPO_ROOT / 'data/metadata/edss_employment_balanced_panel_sensitivity.csv'
JSON_PATH = REPO_ROOT / 'data/metadata/edss_employment_balanced_panel_sensitivity.json'

## Data

제한 DuckDB의 최종 학교–코호트 마트를 읽기 전용으로 다시 집계한다. 학교 OpenID는 균형 패널 자격 판정에만 사용하며 출력하지 않는다.

In [2]:
connection = duckdb.connect(str(DB), read_only=True)
try:
    source_quality = balanced_panel.query_source_quality(connection)
    balanced_panel.validate_source_quality(source_quality)
    aggregates = balanced_panel.query_aggregates(connection)
    composition = balanced_panel.query_composition_counts(connection)
finally:
    connection.close()
rows = balanced_panel.build_sensitivity_rows(aggregates, composition)
by_year = {row['employment_cohort_year']: row for row in rows}
{'source_quality': source_quality, 'cohort_count': len(rows), 'balanced_schools_june': by_year['2010']['balanced_school_count'], 'balanced_schools_december': by_year['2014']['balanced_school_count']}

{'source_quality': {'row_count': 5969,
  'blank_open_id_row_count': 0,
  'duplicate_school_cohort_key_count': 0,
  'cohort_count': 11,
  'first_cohort_year': '2010',
  'last_cohort_year': '2020'},
 'cohort_count': 11,
 'balanced_schools_june': 496,
 'balanced_schools_december': 515}

## Results

전체 표본과 균형 패널의 설명적 비율 및 차이를 코호트별로 대조한다. 2013→2014는 기준일이 달라 변화값을 계산하지 않는다.

In [3]:
lines = [
    '| 코호트 | 전체 학교 | 균형 학교 | 졸업생 포괄률 | 전체 비율 | 균형 비율 | 차이 |',
    '|---:|---:|---:|---:|---:|---:|---:|',
]
for row in rows:
    lines.append(
        f"| {row['employment_cohort_year']} | {row['all_available_school_count']:,} | "
        f"{row['balanced_school_count']:,} | {row['balanced_graduate_coverage_share']:.2%} | "
        f"{row['all_available_reported_employed_share_of_graduates']:.2%} | "
        f"{row['balanced_reported_employed_share_of_graduates']:.2%} | "
        f"{row['balanced_minus_all_reported_employed_share_pp']:+.3f}%p |"
    )
display(Markdown('\n'.join(lines)))

| 코호트 | 전체 학교 | 균형 학교 | 졸업생 포괄률 | 전체 비율 | 균형 비율 | 차이 |
|---:|---:|---:|---:|---:|---:|---:|
| 2010 | 513 | 496 | 98.76% | 49.53% | 49.51% | -0.020%p |
| 2011 | 543 | 496 | 97.94% | 52.24% | 51.95% | -0.288%p |
| 2012 | 543 | 496 | 98.21% | 52.07% | 51.83% | -0.237%p |
| 2013 | 537 | 496 | 98.35% | 51.28% | 50.98% | -0.295%p |
| 2014 | 552 | 515 | 96.06% | 54.25% | 54.03% | -0.217%p |
| 2015 | 554 | 515 | 96.69% | 54.76% | 54.63% | -0.124%p |
| 2016 | 557 | 515 | 97.47% | 54.84% | 54.71% | -0.128%p |
| 2017 | 552 | 515 | 97.76% | 53.18% | 53.03% | -0.147%p |
| 2018 | 544 | 515 | 97.95% | 53.95% | 53.80% | -0.155%p |
| 2019 | 537 | 515 | 98.24% | 52.91% | 52.79% | -0.123%p |
| 2020 | 537 | 515 | 98.25% | 51.25% | 51.14% | -0.108%p |

In [4]:
summary = json.loads(JSON_PATH.read_text(encoding='utf-8'))
findings = summary['findings']
assert findings['direction_agreement_count'] == findings['comparable_transition_count'] == 9
assert abs(findings['balanced_2016_to_2020_change_pp'] + 3.564585) < 1e-9
assert abs(findings['sensitivity_difference_pp'] - 0.020191) < 1e-9
findings

{'all_available_2016_to_2020_change_pp': -3.584776,
 'balanced_2016_to_2020_change_pp': -3.564585,
 'sensitivity_difference_pp': 0.020191,
 'direction_agreement_count': 9,
 'comparable_transition_count': 9,
 'maximum_absolute_balanced_minus_all_gap_pp': 0.29517,
 'maximum_gap_cohort_year': '2013'}

## Takeaways

학교 구성 변화는 2016→2020 하락을 설명하지 못한다. 다만 균형 패널은 구간 전체에 생존한 학교만 포함하므로 생존편향이 생길 수 있고, 시기별 지표 정의 변화까지 해결하지는 않는다. 다음 단계는 동일한 구간·가중 규칙을 유지한 학교 유형·지역별 층화 분석이다.

In [5]:
with CSV_PATH.open(encoding='utf-8', newline='') as handle:
    committed = list(csv.DictReader(handle))
csv_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
assert len(committed) == len(rows) == summary['output']['row_count']
assert tuple(row['employment_cohort_year'] for row in committed) == balanced_panel.EXPECTED_COHORTS
assert committed[0]['previous_comparable_cohort_year'] == ''
assert committed[4]['previous_comparable_cohort_year'] == ''
assert csv_sha256 == summary['output']['sha256']
{'status': 'passed', 'csv_sha256': csv_sha256, 'validation': summary['validation']}

{'status': 'passed',
 'csv_sha256': '4aa9fda9d0298c4e2ba840ac843d95c61fac8b4ce75030dd0da7baeabdb46afd',
 'validation': {'source_mart_quality': {'row_count': 5969,
   'blank_open_id_row_count': 0,
   'duplicate_school_cohort_key_count': 0,
   'cohort_count': 11,
   'first_cohort_year': '2010',
   'last_cohort_year': '2020'},
  'expected_cohorts_present': True,
  'interval_start_changes_are_null': True,
  'balanced_school_count_constant_within_interval': True,
  'all_comparable_transition_directions_agree': True}}